# Notebook 01: Data Loading & Cleaning

**Project 4 — End-to-End Machine Learning: LendingClub Loan Default Prediction**

This notebook handles:
1. Loading the raw LendingClub accepted loans dataset
2. Filtering the target variable (`loan_status`) to terminal outcomes
3. Removing leakage, ID, free-text, high-null, and redundant columns
4. Fixing column formats (percentages, dates, categorical text)
5. Handling missing values
6. Saving the cleaned dataset to parquet

**Output:** `data/cleaned.parquet`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)

print("Libraries loaded.")

Libraries loaded.


## 1. Load Raw Data

In [2]:
df = pd.read_csv('data/accepted_2007_to_2018Q4.csv', low_memory=False)
print(f"Raw dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head(3)

Raw dataset shape: (2260701, 151)
Rows: 2,260,701 | Columns: 151


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,...,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,truck driver,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,home_improvement,NaN,605xx,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Null percentage per column
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Columns with >40% nulls:")
print(null_pct[null_pct > 40].to_string())
print(f"\nTotal columns with >40% nulls: {(null_pct > 40).sum()}")

Columns with >40% nulls:
member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_end_date                              99.517097
hardship_start_date                            99.517097
hardship_type                                  99.517097
hardship_reason                                99.517097
hardship_status                                99.517097
deferral_term                                  99.517097
hardship_last_payment_amount                   99.517097
hardship_payoff_balance_amount                 99.517097
hardship_loan_status                           99.517097
hardship_dpd                                   99.517097
hardship_length                                99.517097
payment_plan_start_date                        99.517097
hardship_amount                                99.517097
settlement_term                                98.485160
debt_settlement_flag_date                      98.485160
settle

In [4]:
df.dtypes.value_counts()

float64    113
str         38
Name: count, dtype: int64

## 2. Filter Target Variable

We keep only terminal loan outcomes:
- **Fully Paid** → 0 (non-default)
- **Charged Off** → 1 (default)

All other statuses (Current, Late, In Grace Period, Default, Issued) are dropped because their final outcome is unknown.

In [5]:
print("loan_status value counts:")
print(df['loan_status'].value_counts())
print(f"\nTotal rows before filter: {len(df):,}")

loan_status value counts:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

Total rows before filter: 2,260,701


In [6]:
# Keep only terminal outcomes
terminal_statuses = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(terminal_statuses)].copy()

# Map to binary target
df['loan_status'] = df['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1})

print(f"Rows after filter: {len(df):,}")
print(f"\nTarget distribution:")
print(df['loan_status'].value_counts())
print(f"\nDefault rate: {df['loan_status'].mean():.2%}")

Rows after filter: 1,345,310

Target distribution:
loan_status
0    1076751
1     268559
Name: count, dtype: int64

Default rate: 19.96%


## 3. Drop Columns

We remove columns in five categories:
1. **Data leakage** — post-origination information that encodes the loan outcome
2. **ID / zero-info** — identifiers and single-value columns
3. **Free-text** — high-cardinality text fields already captured by other columns
4. **Post-2015-only** — columns with 100% nulls for pre-2015 loans
5. **Joint application / high-null / redundant** — sparse or duplicate information

In [7]:
cols_before = df.shape[1]

# 1. Data leakage: post-origination columns
leakage_cols = [
    'funded_amnt', 'funded_amnt_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'last_pymnt_amnt', 'last_pymnt_d',
    'out_prncp', 'out_prncp_inv',
    'collection_recovery_fee', 'recoveries',
    'last_credit_pull_d', 'last_fico_range_low', 'last_fico_range_high',
    'next_pymnt_d', 'disbursement_method',
    'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount',
    'settlement_percentage', 'settlement_term',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date',
    'hardship_end_date', 'payment_plan_start_date', 'hardship_length',
    'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
]

# 2. ID / zero-info
id_cols = ['id', 'member_id', 'url', 'policy_code', 'pymnt_plan']

# 3. Free-text / high-cardinality
text_cols = ['desc', 'title', 'emp_title']

# 4. Post-2015-only columns (100% null for pre-2015 loans)
post2015_cols = [
    'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m',
    'mths_since_rcnt_il', 'total_bal_il', 'il_util',
    'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
    'inq_fi', 'total_cu_tl', 'inq_last_12m',
]

# 5. Joint application columns
joint_cols = [
    'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    'revol_bal_joint',
    'sec_app_fico_range_low', 'sec_app_fico_range_high',
    'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths',
    'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util',
    'sec_app_open_act_il', 'sec_app_num_rev_accts',
    'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog',
]

# 6. High-null columns
high_null_cols = ['mths_since_last_record', 'mths_since_last_major_derog']

# 7. Redundant (keep grade and sub_grade as strings for EDA readability)
redundant_cols = ['zip_code']

# Combine all columns to drop
all_drop_cols = (leakage_cols + id_cols + text_cols + post2015_cols
                 + joint_cols + high_null_cols + redundant_cols)

# Only drop columns that actually exist in the dataframe
existing_drop = [c for c in all_drop_cols if c in df.columns]
missing_drop = [c for c in all_drop_cols if c not in df.columns]

df = df.drop(columns=existing_drop)

print(f"Columns before: {cols_before}")
print(f"Columns dropped: {len(existing_drop)}")
print(f"Columns remaining: {df.shape[1]}")
if missing_drop:
    print(f"\nNote: {len(missing_drop)} listed columns not found in data: {missing_drop}")

Columns before: 151
Columns dropped: 81
Columns remaining: 70


In [8]:
# Check remaining columns and their null rates
remaining_nulls = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Remaining columns with nulls:")
print(remaining_nulls[remaining_nulls > 0].to_string())
print(f"\nRemaining columns: {list(df.columns)}")

Remaining columns with nulls:
mths_since_recent_bc_dlq          76.286506
mths_since_recent_revol_delinq    66.553285
mths_since_last_delinq            50.452535
mths_since_recent_inq             12.939100
num_tl_120dpd_2m                   8.726688
mo_sin_old_il_acct                 7.847634
emp_length                         5.835904
pct_tl_nvr_dlq                     5.030885
avg_cur_bal                        5.021073
mo_sin_old_rev_tl_op               5.019512
mo_sin_rcnt_rev_tl_op              5.019512
num_rev_accts                      5.019512
total_il_high_credit_limit         5.019438
num_accts_ever_120_pd              5.019438
total_rev_hi_lim                   5.019438
mo_sin_rcnt_tl                     5.019438
num_actv_rev_tl                    5.019438
num_actv_bc_tl                     5.019438
tot_coll_amt                       5.019438
num_bc_tl                          5.019438
num_il_tl                          5.019438
num_op_rev_tl                      5.019438
nu

## 4. Format Fixes

Several columns need type conversion:
- `int_rate`, `revol_util`: strip `%` sign → float
- `term`: strip whitespace, extract integer (36 or 60)
- `emp_length`: map text to integer 0–10
- `earliest_cr_line`: parse date, compute credit history length in months
- `issue_d`: parse date (kept for temporal train/test split, dropped before modeling)

**Note:** `grade` and `sub_grade` are kept as original strings (A–G, A1–G5) for EDA readability. Ordinal encoding is deferred to Notebook 03 (Feature Engineering).

In [9]:
# --- int_rate: strip '%' and convert to float ---
df['int_rate'] = df['int_rate'].astype(str).str.strip().str.rstrip('%').astype(float)
print(f"int_rate — dtype: {df['int_rate'].dtype}, range: [{df['int_rate'].min():.2f}, {df['int_rate'].max():.2f}]")

# --- revol_util: strip '%' and convert to float ---
df['revol_util'] = pd.to_numeric(df['revol_util'].astype(str).str.strip().str.rstrip('%'), errors='coerce')
print(f"revol_util — dtype: {df['revol_util'].dtype}, range: [{df['revol_util'].min():.2f}, {df['revol_util'].max():.2f}]")

int_rate — dtype: float64, range: [5.31, 30.99]


revol_util — dtype: float64, range: [0.00, 892.30]


In [10]:
# --- term: extract integer ---
df['term'] = df['term'].astype(str).str.strip().str.extract(r'(\d+)').astype(int)
print(f"term — unique values: {sorted(df['term'].unique())}")

term — unique values: [np.int64(36), np.int64(60)]


In [11]:
# --- emp_length: map text to integer 0-10 ---
print(f"emp_length unique values before: {df['emp_length'].unique()}")

emp_length_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10,
}
df['emp_length'] = df['emp_length'].map(emp_length_map)
print(f"emp_length — dtype: {df['emp_length'].dtype}, nulls: {df['emp_length'].isnull().sum():,}")

emp_length unique values before: <ArrowStringArray>
['10+ years',   '3 years',   '4 years',   '6 years',   '7 years',   '8 years',
   '2 years',   '5 years',   '9 years',  '< 1 year',    '1 year',         nan]
Length: 12, dtype: str
emp_length — dtype: float64, nulls: 78,511


In [12]:
# --- issue_d: parse Mon-YYYY to datetime ---
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
print(f"issue_d — range: [{df['issue_d'].min()} to {df['issue_d'].max()}]")

# --- earliest_cr_line: parse and compute credit age in months ---
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
df['credit_age_months'] = ((df['issue_d'] - df['earliest_cr_line']).dt.days / 30.44).round().astype('Int64')
print(f"credit_age_months — range: [{df['credit_age_months'].min()}, {df['credit_age_months'].max()}]")

# Drop the raw earliest_cr_line (replaced by credit_age_months)
df = df.drop(columns=['earliest_cr_line'])

issue_d — range: [2007-06-01 00:00:00 to 2018-12-01 00:00:00]


credit_age_months — range: [36, 999]


## 5. Handle Missing Values

Three-tier strategy to maximize data retention:

1. **Sentinel fill** — "months since X" columns where NaN genuinely means "event never happened," not missing data. Fill with `max + 1`.
2. **Direct fill** — `emp_length` NaN → 0 (unknown employment); columns with <1% nulls → median/mode.
3. **Median fill** — remaining columns with 3–13% nulls filled with column median. Simple, fast, and preserves all 1.35M rows.

In [13]:
# Current null situation
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100)
null_summary = pd.DataFrame({
    'null_count': null_counts[null_counts > 0],
    'null_pct': null_pct[null_counts > 0].round(2)
}).sort_values('null_pct', ascending=False)
print("Columns with nulls:")
print(null_summary.to_string())

Columns with nulls:
                                null_count  null_pct
mths_since_recent_bc_dlq           1026290     76.29
mths_since_recent_revol_delinq      895348     66.55
mths_since_last_delinq              678743     50.45
mths_since_recent_inq               174071     12.94
num_tl_120dpd_2m                    117401      8.73
mo_sin_old_il_acct                  105575      7.85
emp_length                           78511      5.84
pct_tl_nvr_dlq                       67681      5.03
tot_cur_bal                          67527      5.02
num_accts_ever_120_pd                67527      5.02
tot_hi_cred_lim                      67527      5.02
num_tl_op_past_12m                   67527      5.02
num_tl_90g_dpd_24m                   67527      5.02
num_tl_30dpd                         67527      5.02
num_rev_tl_bal_gt_0                  67527      5.02
num_rev_accts                        67528      5.02
num_op_rev_tl                        67527      5.02
num_il_tl                 

In [14]:
# =============================================
# TIER 1: Sentinel fill for "months since" cols
# NaN = "event never happened", not missing data
# =============================================
sentinel_cols = [
    'mths_since_last_delinq',
    'mths_since_recent_bc_dlq',
    'mths_since_recent_revol_delinq',
    'mths_since_recent_inq',
]

for col in sentinel_cols:
    if col in df.columns:
        sentinel = df[col].max() + 1
        null_ct = df[col].isnull().sum()
        df[col] = df[col].fillna(sentinel)
        print(f"  {col}: filled {null_ct:,} NaNs with sentinel {sentinel:.0f}")

print()

# =============================================
# TIER 2: Direct fill for known-meaning NaNs
# and columns with very low null rates (<1%)
# =============================================

# emp_length: NaN = unknown/not reported → 0
df['emp_length'] = df['emp_length'].fillna(0)
print("  emp_length: filled NaN with 0")

# Columns with <1% nulls → median (numeric) or mode (categorical)
low_null_cols = [col for col in df.columns
                 if 0 < df[col].isnull().sum() / len(df) < 0.01]

for col in low_null_cols:
    null_ct = df[col].isnull().sum()
    if df[col].dtype in ['float64', 'int64', 'Int64']:
        fill_val = df[col].median()
        df[col] = df[col].fillna(fill_val)
        print(f"  {col}: filled {null_ct:,} NaNs with median {fill_val:.2f}")
    else:
        fill_val = df[col].mode()[0]
        df[col] = df[col].fillna(fill_val)
        print(f"  {col}: filled {null_ct:,} NaNs with mode '{fill_val}'")

  mths_since_last_delinq: filled 678,743 NaNs with sentinel 227
  mths_since_recent_bc_dlq: filled 1,026,290 NaNs with sentinel 203
  mths_since_recent_revol_delinq: filled 895,348 NaNs with sentinel 203
  mths_since_recent_inq: filled 174,071 NaNs with sentinel 26

  emp_length: filled NaN with 0
  dti: filled 374 NaNs with median 17.61
  inq_last_6mths: filled 1 NaNs with median 0.00
  revol_util: filled 857 NaNs with median 52.20
  collections_12_mths_ex_med: filled 56 NaNs with median 0.00


  chargeoff_within_12_mths: filled 56 NaNs with median 0.00
  pub_rec_bankruptcies: filled 697 NaNs with median 0.00
  tax_liens: filled 39 NaNs with median 0.00


In [15]:
# =============================================
# TIER 3: Median fill for remaining nulls
# (columns with ~3-13% missing values)
# =============================================
remaining_null_cols = [col for col in df.columns if df[col].isnull().sum() > 0]
print(f"Columns remaining with nulls for median fill: {len(remaining_null_cols)}")
for col in remaining_null_cols:
    pct = df[col].isnull().sum() / len(df) * 100
    print(f"  {col}: {df[col].isnull().sum():,} ({pct:.1f}%)")

for col in remaining_null_cols:
    null_ct = df[col].isnull().sum()
    if df[col].dtype in ['float64', 'int64', 'Int64']:
        fill_val = df[col].median()
        df[col] = df[col].fillna(fill_val)
        print(f"  {col}: filled {null_ct:,} NaNs with median {fill_val:.2f}")
    else:
        fill_val = df[col].mode()[0]
        df[col] = df[col].fillna(fill_val)
        print(f"  {col}: filled {null_ct:,} NaNs with mode '{fill_val}'")

print(f"\nRemaining nulls: {df.isnull().sum().sum()}")

Columns remaining with nulls for median fill: 33
  tot_coll_amt: 67,527 (5.0%)
  tot_cur_bal: 67,527 (5.0%)
  total_rev_hi_lim: 67,527 (5.0%)
  acc_open_past_24mths: 47,281 (3.5%)
  avg_cur_bal: 67,549 (5.0%)
  bc_open_to_buy: 61,143 (4.5%)
  bc_util: 61,912 (4.6%)
  mo_sin_old_il_acct: 105,575 (7.8%)
  mo_sin_old_rev_tl_op: 67,528 (5.0%)
  mo_sin_rcnt_rev_tl_op: 67,528 (5.0%)
  mo_sin_rcnt_tl: 67,527 (5.0%)
  mort_acc: 47,281 (3.5%)
  mths_since_recent_bc: 60,221 (4.5%)
  num_accts_ever_120_pd: 67,527 (5.0%)
  num_actv_bc_tl: 67,527 (5.0%)
  num_actv_rev_tl: 67,527 (5.0%)
  num_bc_sats: 55,841 (4.2%)
  num_bc_tl: 67,527 (5.0%)
  num_il_tl: 67,527 (5.0%)
  num_op_rev_tl: 67,527 (5.0%)
  num_rev_accts: 67,528 (5.0%)
  num_rev_tl_bal_gt_0: 67,527 (5.0%)
  num_sats: 55,841 (4.2%)
  num_tl_120dpd_2m: 117,401 (8.7%)
  num_tl_30dpd: 67,527 (5.0%)
  num_tl_90g_dpd_24m: 67,527 (5.0%)
  num_tl_op_past_12m: 67,527 (5.0%)
  pct_tl_nvr_dlq: 67,681 (5.0%)
  percent_bc_gt_75: 61,555 (4.6%)
  tot_hi_

  tot_cur_bal: filled 67,527 NaNs with median 80231.00
  total_rev_hi_lim: filled 67,527 NaNs with median 24100.00


  acc_open_past_24mths: filled 47,281 NaNs with median 4.00


  avg_cur_bal: filled 67,549 NaNs with median 7407.00
  bc_open_to_buy: filled 61,143 NaNs with median 4700.00
  bc_util: filled 61,912 NaNs with median 63.20
  mo_sin_old_il_acct: filled 105,575 NaNs with median 129.00
  mo_sin_old_rev_tl_op: filled 67,528 NaNs with median 164.00
  mo_sin_rcnt_rev_tl_op: filled 67,528 NaNs with median 8.00
  mo_sin_rcnt_tl: filled 67,527 NaNs with median 5.00
  mort_acc: filled 47,281 NaNs with median 1.00


  mths_since_recent_bc: filled 60,221 NaNs with median 13.00
  num_accts_ever_120_pd: filled 67,527 NaNs with median 0.00
  num_actv_bc_tl: filled 67,527 NaNs with median 3.00
  num_actv_rev_tl: filled 67,527 NaNs with median 5.00


  num_bc_sats: filled 55,841 NaNs with median 4.00
  num_bc_tl: filled 67,527 NaNs with median 7.00
  num_il_tl: filled 67,527 NaNs with median 7.00
  num_op_rev_tl: filled 67,527 NaNs with median 7.00


  num_rev_accts: filled 67,528 NaNs with median 13.00


  num_rev_tl_bal_gt_0: filled 67,527 NaNs with median 5.00


  num_sats: filled 55,841 NaNs with median 11.00
  num_tl_120dpd_2m: filled 117,401 NaNs with median 0.00
  num_tl_30dpd: filled 67,527 NaNs with median 0.00
  num_tl_90g_dpd_24m: filled 67,527 NaNs with median 0.00
  num_tl_op_past_12m: filled 67,527 NaNs with median 2.00
  pct_tl_nvr_dlq: filled 67,681 NaNs with median 98.00
  percent_bc_gt_75: filled 61,555 NaNs with median 42.90
  tot_hi_cred_lim: filled 67,527 NaNs with median 112373.00
  total_bal_ex_mort: filled 47,281 NaNs with median 37296.00
  total_bc_limit: filled 47,281 NaNs with median 15100.00


  total_il_high_credit_limit: filled 67,527 NaNs with median 31681.00

Remaining nulls: 0


## 6. Final Validation & Save

In [16]:
# Final validation — confirm zero nulls without dropping any rows
assert df.isnull().sum().sum() == 0, f"Still have {df.isnull().sum().sum()} nulls!"

print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"Null values: {df.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df['loan_status'].value_counts())
print(f"\nDefault rate: {df['loan_status'].mean():.2%}")
print(f"\nColumn dtypes:")
print(df.dtypes.value_counts())
print(f"\nColumns ({df.shape[1]}):")
print(list(df.columns))

FINAL DATASET SUMMARY
Shape: (1345310, 70)


Null values: 0

Target distribution:
loan_status
0    1076751
1     268559
Name: count, dtype: int64

Default rate: 19.96%

Column dtypes:
float64           58
str                8
int64              2
datetime64[us]     1
Int64              1
Name: count, dtype: int64

Columns (70):
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mth

In [17]:
df.describe()

,loan_amnt,term,int_rate,installment,emp_length,annual_inc,issue_d,loan_status,dti,delinq_2yrs,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,...,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,credit_age_months
count,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1345310,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,...,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1.345310e+06,1345310.0
mean,1.441997e+04,4.179020e+01,1.323962e+01,4.380755e+02,5.617710e+00,7.624764e+04,2015-06-06 17:48:20.835049,1.996261e-01,1.828248e+01,3.177944e-01,6.961850e+02,7.001852e+02,6.550802e-01,1.315280e+02,1.159352e+01,2.152760e-01,1.624811e+04,5.181027e+01,2.498084e+01,1.713285e-02,5.045677e-03,2.368682e+02,1.380768e+05,3.235470e+04,4.668519e+00,1.318325e+04,9.936890e+03,6.008962e+01,9.061852e-03,1.489553e+01,...,1.285891e+01,7.711460e+00,1.647208e+00,2.333350e+01,1.642646e+02,9.208746e+00,1.470787e+02,4.839963e-01,3.610470e+00,5.611183e+00,4.703749e+00,8.045836e+00,8.489191e+00,8.214660e+00,1.452951e+01,5.562908e+00,1.161255e+01,7.485264e-04,3.246835e-03,8.442218e-02,2.169908e+00,9.435762e+01,4.505276e+01,1.343742e-01,5.212851e-02,1.712751e+05,4.923159e+04,2.140436e+04,4.160855e+04,195.112739
min,5.000000e+02,3.600000e+01,5.310000e+00,4.930000e+00,0.000000e+00,0.000000e+00,2007-06-01 00:00:00,0.000000e+00,-1.000000e+00,0.000000e+00,6.250000e+02,6.290000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,36.0
25%,8.000000e+03,3.600000e+01,9.750000e+00,2.484800e+02,2.000000e+00,4.578000e+04,2014-07-01 00:00:00,0.000000e+00,1.179000e+01,0.000000e+00,6.700000e+02,6.740000e+02,0.000000e+00,3.100000e+01,8.000000e+00,0.000000e+00,5.943000e+03,3.350000e+01,1.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00,3.088900e+04,1.450000e+04,2.000000e+00,3.238000e+03,1.574000e+03,3.960000e+01,0.000000e+00,0.000000e+00,...,4.000000e+00,3.000000e+00,0.000000e+00,6.000000e+00,2.030000e+02,2.000000e+00,5.200000e+01,0.000000e+00,2.000000e+00,3.000000e+00,3.000000e+00,5.000000e+00,4.000000e+00,5.000000e+00,9.000000e+00,3.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,9.190000e+01,1.250000e+01,0.000000e+00,0.00000

In [18]:
# Save cleaned dataset
df.to_parquet('data/cleaned.parquet', index=False)
print(f"Saved cleaned dataset to data/cleaned.parquet")
print(f"File size: {pd.io.common.file_exists('data/cleaned.parquet')}")

# Quick verification: reload and check
df_check = pd.read_parquet('data/cleaned.parquet')
print(f"Reload verification — shape: {df_check.shape}, nulls: {df_check.isnull().sum().sum()}")
del df_check

Saved cleaned dataset to data/cleaned.parquet
File size: True


Reload verification — shape: (1345310, 70), nulls: 0
